In [1]:
!pip install nltk gensim sentencepiece transformers tokenizers pandas numpy scikit-learn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 50.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

from gensim.models import Word2Vec, FastText
import gensim.downloader as api

from transformers import BertTokenizer

import sentencepiece as spm

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [6]:
df = pd.read_csv("IMDB Dataset.csv")

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [9]:
print(df.columns)
TEXT_COLUMN = "review"

Index(['review', 'sentiment'], dtype='object')


In [10]:
text = " ".join(df[TEXT_COLUMN].astype(str).head(100))

In [11]:
text_lower = text.lower()

print(text_lower[:500])

one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.<br /><br />it is called oz


In [12]:
sentences = sent_tokenize(text_lower)

print("Number of sentences:", len(sentences))
print(sentences[:5])

Number of sentences: 935
["one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked.", 'they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go.', 'trust me, this is not a show for the faint hearted or timid.', 'this show pulls no punches with regards to drugs, sex or violence.', 'its is hardcore, in the classic use of the word.<br /><br />it is called oz as that is the nickname given to the oswald maximum security state penitentary.']


In [13]:
stop_words = set(stopwords.words('english'))

processed_sentences = []

for sentence in sentences:
    words = word_tokenize(sentence)

    filtered = [
        word for word in words
        if word.isalpha() and word not in stop_words
    ]

    processed_sentences.append(filtered)

processed_sentences[:3]

[['one', 'reviewers', 'mentioned', 'watching', 'oz', 'episode', 'hooked'],
 ['right',
  'exactly',
  'happened',
  'br',
  'br',
  'first',
  'thing',
  'struck',
  'oz',
  'brutality',
  'unflinching',
  'scenes',
  'violence',
  'set',
  'right',
  'word',
  'go'],
 ['trust', 'show', 'faint', 'hearted', 'timid']]

In [14]:
stemmer = PorterStemmer()

stemming_results = []

for word in processed_sentences[0][:20]:
    stemming_results.append(
        (word, stemmer.stem(word))
    )

pd.DataFrame(
    stemming_results,
    columns=["Original","Stemmed"]
)

,Original,Stemmed
0,one,one
1,reviewers,review
2,mentioned,mention
3,watching,watch
4,oz,oz
5,episode,episod
6,hooked,hook


In [15]:
lemmatizer = WordNetLemmatizer()

lemmatization_results = []

for word in processed_sentences[0][:20]:
    lemmatization_results.append(
        (word, lemmatizer.lemmatize(word))
    )

pd.DataFrame(
    lemmatization_results,
    columns=["Original","Lemmatized"]
)

,Original,Lemmatized
0,one,one
1,reviewers,reviewer
2,mentioned,mentioned
3,watching,watching
4,oz,oz
5,episode,episode
6,hooked,hooked


In [16]:
word2vec_model = Word2Vec(
    sentences=processed_sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [17]:
embedding_matrix = word2vec_model.wv.vectors

print("Embedding Matrix Shape:")
print(embedding_matrix.shape)

Embedding Matrix Shape:
(4400, 100)


In [18]:
embedding_matrix[:5]

array([[-7.23724579e-03,  8.40644352e-03,  8.44744686e-03,
         1.17207179e-02, -6.20485796e-03, -2.05195490e-02,
         1.01302024e-02,  2.64075510e-02, -9.97071620e-03,
        -8.27547908e-03,  1.53340551e-03, -1.61417928e-02,
        -5.47811575e-03,  1.04558645e-02, -3.48402769e-03,
        -9.21717566e-03,  4.26336611e-03, -9.37475264e-03,
        -6.53802650e-03, -2.58429181e-02,  1.45954117e-02,
         3.89924156e-03,  1.28759276e-02, -6.25773473e-03,
         7.42262136e-03, -3.61372204e-03, -5.91621781e-03,
         2.18992797e-03, -1.54170869e-02, -3.36398766e-03,
         2.28878180e-03, -1.81009222e-04,  1.36271259e-02,
        -1.42694348e-02, -3.52662080e-03,  5.41201001e-03,
         8.29120167e-03, -1.16081862e-02, -4.96572349e-03,
        -1.68629847e-02, -9.66308080e-03, -3.04624834e-03,
        -1.24529544e-02, -3.27530480e-03,  6.16783462e-03,
        -6.10516453e-03, -1.74416583e-02,  9.24719963e-03,
         7.87667930e-03,  1.61331296e-02, -5.77295944e-0

In [19]:
word = word2vec_model.wv.index_to_key[0]

print("Word:", word)

print(
    word2vec_model.wv[word]
)

Word: br
[-0.00723725  0.00840644  0.00844745  0.01172072 -0.00620486 -0.02051955
  0.0101302   0.02640755 -0.00997072 -0.00827548  0.00153341 -0.01614179
 -0.00547812  0.01045586 -0.00348403 -0.00921718  0.00426337 -0.00937475
 -0.00653803 -0.02584292  0.01459541  0.00389924  0.01287593 -0.00625773
  0.00742262 -0.00361372 -0.00591622  0.00218993 -0.01541709 -0.00336399
  0.00228878 -0.00018101  0.01362713 -0.01426943 -0.00352662  0.00541201
  0.0082912  -0.01160819 -0.00496572 -0.01686298 -0.00966308 -0.00304625
 -0.01245295 -0.0032753   0.00616783 -0.00610516 -0.01744166  0.0092472
  0.00787668  0.01613313 -0.00577296  0.00026185 -0.00814707  0.00013028
  0.00183226 -0.00063175  0.01274545 -0.00661443 -0.01073102  0.01276977
  0.00276576  0.00229147 -0.01034168 -0.00789567 -0.0107875   0.01033216
 -0.00072733  0.01148865 -0.01072367  0.01272901  0.00206585  0.01181001
  0.0090918  -0.01322835  0.0147584   0.0068193   0.00962355 -0.0027284
 -0.01096565 -0.00777243 -0.00536711  0.0013

In [21]:
word2vec_model.wv.most_similar(
    positive=[word],
    topn=5
)

[('story', 0.5523805618286133),
 ('time', 0.5145125985145569),
 ('film', 0.5089575052261353),
 ('movie', 0.5008211135864258),
 ('acting', 0.4925498068332672)]

In [22]:
glove = api.load(
    "glove-wiki-gigaword-100"
)

[==================================================] 100.0% 128.1/128.1MB downloaded


In [23]:
glove['computer'][:10]

array([-0.16298 ,  0.30141 ,  0.57978 ,  0.066548,  0.45835 , -0.15329 ,
        0.43258 , -0.89215 ,  0.57747 ,  0.36375 ], dtype=float32)

In [24]:
fasttext_model = FastText(
    sentences=processed_sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [25]:
fasttext_model.wv["computer"][:10]

array([-0.01750059,  0.0154982 , -0.02062754,  0.00117601,  0.02015363,
        0.01628401, -0.01145323,  0.00807036,  0.02281672, -0.02552935],
      dtype=float32)

In [26]:
comparison_word = "computer"

results = {
    "Word2Vec":
        word2vec_model.wv[comparison_word][:10]
        if comparison_word in word2vec_model.wv
        else "Not Found",

    "GloVe":
        glove[comparison_word][:10]
        if comparison_word in glove
        else "Not Found",

    "FastText":
        fasttext_model.wv[comparison_word][:10]
}

In [27]:
results

{'Word2Vec': array([ 0.00441563,  0.00555598, -0.00398549, -0.00424531,  0.00195353,
        -0.0049823 ,  0.00543357, -0.00390421, -0.00794297, -0.01014297],
       dtype=float32),
 'GloVe': array([-0.16298 ,  0.30141 ,  0.57978 ,  0.066548,  0.45835 , -0.15329 ,
         0.43258 , -0.89215 ,  0.57747 ,  0.36375 ], dtype=float32),
 'FastText': array([-0.01750059,  0.0154982 , -0.02062754,  0.00117601,  0.02015363,
         0.01628401, -0.01145323,  0.00807036,  0.02281672, -0.02552935],
       dtype=float32)}

In [28]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

bpe_tokenizer = Tokenizer(BPE())
bpe_tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=1000
)

corpus = [
    " ".join(
        [" ".join(x)
         for x in processed_sentences]
    )
]

bpe_tokenizer.train_from_iterator(
    corpus,
    trainer
)

In [29]:
sentence = "unhappiness is increasing rapidly"

bpe_output = bpe_tokenizer.encode(sentence)

print(bpe_output.tokens)

['un', 'happ', 'iness', 'is', 'inc', 're', 'as', 'ing', 'r', 'ap', 'id', 'ly']


In [30]:
bert_tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

sentence = "unhappiness is increasing rapidly"

wordpiece_tokens = bert_tokenizer.tokenize(
    sentence
)

print(wordpiece_tokens)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

['un', '##ha', '##pp', '##iness', 'is', 'increasing', 'rapidly']


In [31]:
with open("corpus.txt","w",
          encoding="utf8") as f:

    for sentence in sentences:
        f.write(sentence + "\n")

In [32]:
spm.SentencePieceTrainer.train(
    input='corpus.txt',
    model_prefix='spm',
    vocab_size=1000
)

True

In [33]:
sp = spm.SentencePieceProcessor()
sp.load("spm.model")

True

In [34]:
sentencepiece_tokens = sp.encode(
    "unhappiness is increasing rapidly",
    out_type=str
)

print(sentencepiece_tokens)

['▁un', 'h', 'a', 'p', 'p', 'i', 'ness', '▁is', '▁in', 'c', 're', 'a', 's', 'ing', '▁ra', 'p', 'id', 'ly']


In [35]:
comparison = pd.DataFrame({
    "BPE":[str(bpe_output.tokens)],
    "WordPiece":[str(wordpiece_tokens)],
    "SentencePiece":[str(sentencepiece_tokens)]
})

comparison

,BPE,WordPiece,SentencePiece
0,"['un', 'happ', 'iness', 'is', 'inc', 're', 'as...","['un', '##ha', '##pp', '##iness', 'is', 'incre...","['▁un', 'h', 'a', 'p', 'p', 'i', 'ness', '▁is'..."


In [36]:
word = "computer"

comparison_embeddings = pd.DataFrame({
    "Word2Vec":
        word2vec_model.wv[word][:10]
        if word in word2vec_model.wv
        else np.zeros(10),

    "GloVe":
        glove[word][:10]
        if word in glove
        else np.zeros(10),

    "FastText":
        fasttext_model.wv[word][:10]
})

In [37]:
comparison_embeddings

,Word2Vec,GloVe,FastText
0,0.004416,-0.162980,-0.017501
1,0.005556,0.301410,0.015498
2,-0.003985,0.579780,-0.020628
3,-0.004245,0.066548,0.001176
4,0.001954,0.458350,0.020154
5,-0.004982,-0.153290,0.016284
6,0.005434,0.432580,-0.011453
7,-0.003904,-0.892150,0.008070
8,-0.007943,0.577470,0.022817
9,-0.010143,0.363750,-0.025529


In [38]:
print("""
1. Text preprocessing completed.
2. Stopwords removed.
3. Stemming and lemmatization applied.
4. Word2Vec embedding matrix generated.
5. GloVe embeddings loaded.
6. FastText embeddings trained.
7. BPE, WordPiece and SentencePiece compared.
8. Static embedding comparison completed.
""")


1. Text preprocessing completed.
2. Stopwords removed.
3. Stemming and lemmatization applied.
4. Word2Vec embedding matrix generated.
5. GloVe embeddings loaded.
6. FastText embeddings trained.
7. BPE, WordPiece and SentencePiece compared.
8. Static embedding comparison completed.

